## Neural Network billi ko kaise dekhta hai

Ek neural network, jo ImageNet par pehle se pre-trained hai, 1000 alag-alag types ke objects ko pehchaan sakta hai, jaise alag-alag breeds ki billiyan. Yeh dekhna interesting hoga ki ek neural network ke liye **ideal Siamese cat** kaisi dikhti hai.

> Bilkul, aap *Siamese cat* ko kisi bhi doosre ImageNet class se replace kar sakte hain.

Shuru karne ke liye, chaliye VGG network load karte hain:

In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
from PIL import Image
import json
np.set_printoptions(precision=3,suppress=True)

model = keras.applications.VGG16(weights='imagenet',include_top=True)
classes = json.loads(open('imagenet_classes.json','r').read())

## Result ke liye Optimization

Ideal billi dekhne ke liye, hum ek random noise wali image se shuru karenge aur gradient descent optimization technique use karke image ko is tarah adjust karne ki koshish karenge ki network ek billi ko pehchaan le.

![Optimization Loop](../../../../../lessons/4-ComputerVision/08-TransferLearning/images/ideal-cat-loop.png)

Yeh hamari starting image hai:

In [ ]:
x = tf.Variable(tf.random.normal((1,224,224,3)))

def normalize(img):
    return (img-tf.reduce_min(img))/(tf.reduce_max(img)-tf.reduce_min(img))

plt.imshow(normalize(x[0]))

Hum `normalize` function use karte hain taaki hamari values 0-1 ki range mein aa jayein.

Agar hum is image par apna VGG network call karte hain, toh hamein probabilities ka kam-ya-zyada random distribution milega:

In [ ]:
def plot_result(x):
    res = model(x)[0]
    cls = tf.argmax(res)
    print(f"Predicted class: {cls} ({classes[cls]})")
    print(f"Probability of predicted class = {res[cls]}")
    fig,ax = plt.subplots(1,2,figsize=(15,2.5),gridspec_kw = { "width_ratios" : [1,5]} )
    ax[0].imshow(normalize(x[0]))
    ax[0].axis('off')
    ax[1].bar(range(1000),res,width=3)
    plt.show()

plot_result(x)

Haalaanki yeh lag sakta hai ki kisi ek class ki probability baaki classes se bahut zyada hai, phir bhi woh bahut kam hai - actual probability dekhne ke liye scale par dhyan dijiye, jo abhi bhi lagbhag 5% ke aaspaas hai.

Ab chaliye ek target class (jaise **Siamese cat**) choose karte hain, aur gradient descent use karke image ko adjust karna shuru karte hain. Agar $x$ input image hai, aur $V$ VGG network hai, toh hum loss function $\mathcal{L} = \mathcal{L}(c,V(x))$ (jahan $c$ target class hai) calculate karenge, aur $x$ ko neeche diye formula se adjust karenge:
$$
x^{(i+1)} = x^{(i)} - \eta{\partial \mathcal{L}\over\partial x}
$$
Loss function cross-entropy loss hoga, kyunki hum do probability distributions ko compare kar rahe hain. Hamare case mein, kyunki class ko ek number se represent kiya gaya hai, na ki one-hot encoded vector se, hum *sparse categorical cross-entropy* use karenge.

Hum is process ko kai epochs tak repeat karenge, aur har step par image print karenge.

> Is code ko GPU-enabled compute par execute karna behtar hai, ya phir waiting time kam karne ke liye epochs ki sankhya ghata dijiye.

In [ ]:
target = [284] # Siamese cat

def cross_entropy_loss(target,res):
    return tf.reduce_mean(keras.metrics.sparse_categorical_crossentropy(target,res))

def optimize(x,target,epochs=1000,show_every=None,loss_fn=cross_entropy_loss, eta=1.0):
    if show_every is None:
        show_every = epochs // 10
    for i in range(epochs):
        with tf.GradientTape() as t:
            res = model(x)
            loss = loss_fn(target,res)
            grads = t.gradient(loss,x)
            x.assign_sub(eta*grads)
            if i%show_every == 0:
                clear_output(wait=True)
                print(f"Epoch: {i}, loss: {loss}")
                plt.imshow(normalize(x[0]))
                plt.show()

optimize(x,target)

In [ ]:
plot_result(x)

Ab hamare paas ek aisi image hai jo ek neural network ko billi jaisi lagti hai, bhale hi hamein woh abhi bhi noise jaisi hi lagti ho. Agar hum ise thodi der aur optimize karein, toh hamein **ideal noisy cat** ki image mil sakti hai, jiski probability 1 ke kareeb hogi.

## Noise ko samajhna

Yeh noise hamein zyada samajh nahi aata, lekin isme shayad bahut saare low-level filters hote hain jo billi ke liye common hain. Haalaanki, kyunki ideal result ke liye input ko optimize karne ke bahut saare tareeke hain, optimization algorithm ko aise patterns dhoondhne ke liye prompt nahi kiya jaata jo visually samajhne layak hon.

Is noise ko thoda kam noise jaisa dikhane ke liye, hum loss function mein ek extra term add kar sakte hain - **variation loss**. Yeh measure karta hai ki image ke neighboring pixels kitne similar hain. Agar hum is term ko apne *loss function* mein add karte hain, toh yeh optimizer ko kam noisy solutions dhoondhne ke liye force karega, aur is tarah zyada recognizable details honge.

> Practice mein, achhe results paane ke liye hamein cross-entropy loss aur variation loss ke beech balance banana padta hai. Apne function mein, hum kuch numeric coefficients add karte hain, aur aap unke saath experiment karke dekh sakte hain ki image kaise change hoti hai.

In [ ]:
def total_loss(target,res):
    return 10*tf.reduce_mean(keras.metrics.sparse_categorical_crossentropy(target,res)) + \
           0.005*tf.image.total_variation(x,res)

optimize(x,target,loss_fn=total_loss)

Yeh hamare neural network ke liye ek billi ki ideal image hai, aur hum kuch jaani-pehchaani features jaise aankhein aur kaan bhi dekh sakte hain. Inme se kaafi saare hain, jo neural network ko aur zyada sure banate hain ki yeh ek billi hai.

In [ ]:
plot_result(x)

Chaliye yeh bhi dekhte hain ki VGG ke liye kuch doosre objects kaise dikhte hain:

In [ ]:
x = tf.Variable(tf.random.normal((1,224,224,3)))
optimize(x,[340],loss_fn=total_loss) # zebra

## Adversarial Attacks

Kyunki *ideal cat* ki image ek random noise jaisi dikh sakti hai, yeh suggest karta hai ki shayad hum kisi bhi image ko thoda sa badal kar uski class change kar sakte hain. Chaliye ise thoda experiment karke dekhte hain. Hum ek kutte ki image se shuru karenge:

In [ ]:
img = Image.open('images/dog-from-unsplash.jpg')
img = img.crop((200,20,600,420)).resize((224,224))
img = np.array(img)
plt.imshow(img)

In [ ]:
plot_result(np.expand_dims(img,axis=0))

In [ ]:
x = tf.Variable(np.expand_dims(img,axis=0).astype(np.float32)/255.0)
optimize(x,target,epochs=100)

In [ ]:
plot_result(x)

Toh, upar di gayi picture VGG network ke point of view se ek perfect billi hai!

## ResNet ke saath experiment karna

Ab dekhte hain ki yahi picture ek alag model, jaise ResNet, dwara kaise classify hoti hai:

In [ ]:
model = keras.applications.ResNet50(weights='imagenet',include_top=True)

> Kyunki humne `model` ko ek global variable ke roop mein use kiya, ab se saare functions VGG ke bajay ResNet use karenge.

In [ ]:
plot_result(x)

Zaahir hai, result kaafi alag hai. Yeh bilkul expected hai, kyunki jab humne ek billi ke liye optimize kiya, tab humne VGG network ki nature, uske low-level filters waghera ko dhyan mein rakha. Kyunki ResNet ke filters alag hain, isliye woh alag result deta hai. Isse hamein ek idea milta hai ki hum khud ko adversarial attacks se kaise bacha sakte hain - alag-alag models ka ensemble use karke.

Chaliye dekhte hain ki ResNet ke liye ideal zebra kaisa dikhta hai:

In [ ]:
x = tf.Variable(tf.random.normal((1,224,224,3)))
optimize(x,target=[340],epochs=500,loss_fn=total_loss)

In [ ]:
plot_result(x)

Yeh picture kaafi alag hai, jo hamein batati hai ki objects ko pehchaanne ke tareeke mein ek neural network ka structure ek important role play kar sakta hai.

> **Task**: ResNet par adversarial attack karne ki koshish kijiye, aur results ko compare kijiye.

## Alag-alag Optimizers use karna

Hamare example mein, humne sabse simple optimization technique - gradient descent - use ki hai. Haalaanki, Keras framework mein alag-alag built-in [optimizers](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers) shaamil hain, aur hum gradient descent ke bajay unhe use kar sakte hain. Iske liye hamare code mein bahut kam changes ki zaroorat hogi - hum us hisse ko badlenge jahan hum input image ko adjust karte hain `x.assign_sub(eta*grads)` ko optimizer ke `apply_gradients` function ke call se:

In [ ]:
def optimize(x,target,epochs=1000,show_every=None,loss_fn=cross_entropy_loss,optimizer=keras.optimizers.SGD(learning_rate=1)):
    if show_every is None:
        show_every = epochs // 10
    for i in range(epochs):
        with tf.GradientTape() as t:
            res = model(x)
            loss = loss_fn(target,res)
            grads = t.gradient(loss,x)
            optimizer.apply_gradients([(grads,x)])
            if i%show_every == 0:
                clear_output(wait=True)
                print(f"Epoch: {i}, loss: {loss}")
                plt.imshow(normalize(x[0]))
                plt.show()

x = tf.Variable(tf.random.normal((1,224,224,3)))

optimize(x,[898],loss_fn=total_loss) # water bottle

## Conclusion

Hum pre-trained CNN ke andar gradient descent optimization use karke ek billi (aur doosre objects) ki ideal image ko visualize kar paaye, jisme humne weights ke bajay input image ko adjust kiya. Ek aisi image paane ki main trick, jo kuch had tak samajh mein aaye, yeh thi ki variation loss ko ek additional loss function ke roop mein use kiya jaye, jo image ko zyada smooth dikhne ke liye force karta hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->